# 🧠 Toy Neural Network — From Scratch with NumPy

A ground-up implementation of a feedforward neural network using **only NumPy** — no PyTorch, no Keras.  
Building every piece by hand is the best way to truly understand deep learning.

## Learning Objectives
By the end of this notebook you will be able to:
1. Generate and visualise a non-linearly-separable 2-D dataset
2. Implement **sigmoid**, **ReLU**, and **tanh** activation functions and their derivatives
3. Build a flexible multi-layer network with **Xavier weight initialisation**
4. Code **forward propagation** layer-by-layer
5. Derive and code **binary cross-entropy loss**
6. Implement **backpropagation** using the chain rule
7. Train the network and **plot the loss curve**
8. Visualise the **decision boundary** learned by the network

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

## §1 — Generate & Visualise the Dataset

We use **`make_moons`** — two interleaving half-circles that a straight line *cannot* separate.  
This forces the network to learn a **non-linear** decision boundary.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Generate dataset ─────────────────────────────────────────
X, y = make_moons(n_samples=600, noise=0.25, random_state=42)
y = y.reshape(-1, 1).astype(float)

# Standardise features (zero mean, unit variance)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train / test split  (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} samples   Test: {X_test.shape[0]} samples")
print(f"Class balance  — train: {y_train.mean():.2f}  test: {y_test.mean():.2f}")

# ── Visualise ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
colors = ['#4878d0', '#ee854a']
for cls, label in [(0, 'Class 0'), (1, 'Class 1')]:
    mask = (y_train.ravel() == cls)
    ax.scatter(X_train[mask, 0], X_train[mask, 1],
               c=colors[cls], label=label, alpha=0.7, edgecolors='white', linewidths=0.4)
ax.set_title('make_moons — training data')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout(); plt.show()

## §2 — Activation Functions & Derivatives

Activation functions introduce **non-linearity**, allowing the network to learn complex patterns.

| Function | Formula | Derivative | Use-case |
|---|---|---|---|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma(z)(1-\sigma(z))$ | Output layer (binary) |
| Tanh | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $1 - \tanh^2(z)$ | Hidden layers |
| ReLU | $\max(0, z)$ | $\mathbb{1}[z > 0]$ | Hidden layers (modern) |

In [ ]:
# ── Activation functions ────────────────────────────────────
def sigmoid(z):      return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def sigmoid_d(z):    s = sigmoid(z); return s * (1 - s)
def tanh_act(z):     return np.tanh(z)
def tanh_d(z):       return 1 - np.tanh(z)**2
def relu(z):         return np.maximum(0, z)
def relu_d(z):       return (z > 0).astype(float)

ACTIVATIONS = {
    'sigmoid': (sigmoid, sigmoid_d),
    'tanh':    (tanh_act, tanh_d),
    'relu':    (relu, relu_d),
}

# ── Plot all three ───────────────────────────────────────────
z = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=False)
for ax, (name, (fn, dfn)) in zip(axes, ACTIVATIONS.items()):
    ax.plot(z, fn(z),  lw=2.5, label=f'{name}(z)')
    ax.plot(z, dfn(z), lw=2,   ls='--', label=f"derivative")
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(name.capitalize()); ax.legend(fontsize=9)
plt.suptitle('Activation Functions & Their Derivatives', fontweight='bold')
plt.tight_layout(); plt.show()

## §3 — Network Architecture & Weight Initialisation

Our network is a fully-connected MLP with configurable hidden layers.

```
Input (2)  →  Hidden 1 (16, tanh)  →  Hidden 2 (8, tanh)  →  Output (1, sigmoid)
```

**Xavier / Glorot initialisation**: sets the initial weight variance to $\frac{2}{n_{in}+n_{out}}$, preventing gradients from vanishing or exploding in early training.

In [ ]:
def init_params(layer_dims):
    """Xavier-initialised weights and zero biases.
    layer_dims: list of ints, e.g. [2, 16, 8, 1]
    Returns: list of dicts with keys 'W' and 'b'.
    """
    params = []
    for i in range(len(layer_dims) - 1):
        n_in, n_out = layer_dims[i], layer_dims[i + 1]
        scale = np.sqrt(2.0 / (n_in + n_out))   # Xavier
        params.append({
            'W': np.random.randn(n_in, n_out) * scale,
            'b': np.zeros((1, n_out))
        })
    return params

# Default architecture: 2 inputs → 16 → 8 → 1 output
ARCH = [2, 16, 8, 1]
params = init_params(ARCH)

print("Layer shapes (W, b):")
for i, p in enumerate(params):
    print(f"  Layer {i+1}: W={p['W'].shape}  b={p['b'].shape}")

## §4 — Forward Propagation

For each layer $l$:
$$z^{[l]} = A^{[l-1]} W^{[l]} + b^{[l]}, \qquad A^{[l]} = g^{[l]}(z^{[l]})$$

Hidden layers use **tanh**; the final layer uses **sigmoid** to produce a probability in $[0, 1]$.

In [ ]:
def forward(X, params, hidden_act='tanh'):
    """Run forward pass.
    Returns: output probabilities (n, 1), and cache of (Z, A) per layer.
    """
    act_fn, _ = ACTIVATIONS[hidden_act]
    cache = []
    A = X
    for i, p in enumerate(params):
        Z = A @ p['W'] + p['b']
        # Last layer → sigmoid; hidden layers → chosen activation
        A = sigmoid(Z) if i == len(params) - 1 else act_fn(Z)
        cache.append((Z, A))
    return A, cache   # A is final output ŷ

# Quick sanity check
y_hat, _ = forward(X_train[:5], params)
print("Output probabilities (first 5 samples):", y_hat.ravel().round(4))

## §5 — Loss Function: Binary Cross-Entropy

For binary classification the standard loss is:

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\bigl[y_i \log\hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\bigr]$$

A small constant $\epsilon = 10^{-8}$ is added inside the log to prevent $\log(0)$.

In [ ]:
eps = 1e-8

def bce_loss(y_true, y_hat):
    """Binary cross-entropy loss."""
    n = y_true.shape[0]
    return -np.mean(y_true * np.log(y_hat + eps) + (1 - y_true) * np.log(1 - y_hat + eps))

def accuracy(y_true, y_hat, threshold=0.5):
    preds = (y_hat >= threshold).astype(float)
    return np.mean(preds == y_true)

# Test on random weights
y_hat_test, _ = forward(X_train, params)
print(f"Initial loss:    {bce_loss(y_train, y_hat_test):.4f}")
print(f"Initial accuracy:{accuracy(y_train, y_hat_test):.4f}  (expect ~0.50)")